## Import Libraries 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## Load the MNIST Data

In [2]:
train_df = pd.read_csv("Data/train.csv")
test_df  = pd.read_csv("Data/test.csv")

X_train = train_df.iloc[:, 1:].values / 255.0
y_train = train_df.iloc[:, 0].values

X_test  = test_df.iloc[:, 1:].values / 255.0
y_test  = test_df.iloc[:, 0].values

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape, y_train_tensor.shape)


torch.Size([42000, 784]) torch.Size([42000])


## Create DataLoaders

In [3]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)


## CNN Model Definition (Configurable)

In [4]:
class CNN(nn.Module):
    def __init__(self, activation=nn.ReLU, dropout=0.25):
        super().__init__()
        self.act = activation()
        
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool  = nn.MaxPool2d(2)
        self.drop  = nn.Dropout(dropout)
        
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.pool(x)
        x = self.drop(x)
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        return self.fc2(x)


## MLP Model Definition (Configurable)

In [5]:
def train_model(model, optimizer, epochs):
    criterion = nn.CrossEntropyLoss()
    train_loss, train_acc = [], []

    for epoch in range(epochs):
        model.train()
        correct, total, running_loss = 0, 0, 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, pred = outputs.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)

        train_loss.append(running_loss / len(train_loader))
        train_acc.append(correct / total)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Loss: {train_loss[-1]:.4f} | "
              f"Acc: {train_acc[-1]:.4f}")

    return train_loss, train_acc


def test_model(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            _, pred = outputs.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)

    return correct / total


## Training & Evaluation Functions

In [6]:
def train_model(model, optimizer, epochs):
    criterion = nn.CrossEntropyLoss()
    train_loss, train_acc = [], []

    for epoch in range(epochs):
        model.train()
        correct, total, running_loss = 0, 0, 0

        for x, y in train_loader:
            # 🔥 THIS WAS MISSING
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, pred = outputs.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)

        train_loss.append(running_loss / len(train_loader))
        train_acc.append(correct / total)

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {train_loss[-1]:.4f} | "
            f"Acc: {train_acc[-1]:.4f}"
        )

    return train_loss, train_acc


def test_model(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            # 🔥 THIS WAS MISSING
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            _, pred = outputs.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)

    return correct / total


## Task 1: Activation Function Challenge (CNN)

In [7]:
activations = {
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
    "ReLU": nn.ReLU
}

task1_results = {}

for name, act in activations.items():
    print(f"\nTraining CNN with {name}")
    model = CNN(activation=act).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    loss, acc = train_model(model, optimizer, epochs=10)
    test_acc = test_model(model)

    task1_results[name] = {
        "loss": loss,
        "acc": acc,
        "test_acc": test_acc
    }



Training CNN with Sigmoid
Epoch 1/10 | Loss: 1.7566 | Acc: 0.3449
Epoch 2/10 | Loss: 0.2415 | Acc: 0.9307
Epoch 3/10 | Loss: 0.1639 | Acc: 0.9526
Epoch 4/10 | Loss: 0.1220 | Acc: 0.9648
Epoch 5/10 | Loss: 0.0923 | Acc: 0.9730
Epoch 6/10 | Loss: 0.0715 | Acc: 0.9797
Epoch 7/10 | Loss: 0.0554 | Acc: 0.9839
Epoch 8/10 | Loss: 0.0463 | Acc: 0.9866
Epoch 9/10 | Loss: 0.0378 | Acc: 0.9896
Epoch 10/10 | Loss: 0.0317 | Acc: 0.9910


RuntimeError: shape '[-1, 1, 28, 28]' is invalid for input of size 50112

## Plot Activation Function Comparison

In [ ]:
plt.figure(figsize=(8,5))
for name in task1_results:
    plt.plot(task1_results[name]["acc"], label=name)

plt.xlabel("Epochs")
plt.ylabel("Training Accuracy")
plt.title("Activation Function Comparison")
plt.legend()
plt.show()


## Task 2: Optimizer Showdown (ReLU fixed)

In [ ]:
optimizers = {
    "SGD": lambda p: optim.SGD(p, lr=0.01),
    "SGD+Momentum": lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam": lambda p: optim.Adam(p, lr=0.001)
}

task2_results = {}

for name, opt_fn in optimizers.items():
    print(f"\nOptimizer: {name}")
    model = CNN(activation=nn.ReLU).to(device)
    optimizer = opt_fn(model.parameters())

    train_model(model, optimizer, epochs=10)
    acc = test_model(model)
    task2_results[name] = acc


## Task 3: BN & Dropout Scenarios (MLP)

In [ ]:
scenarios = {
    "No BN, No Dropout": (False, 0.0),
    "No BN, Dropout=0.1": (False, 0.1),
    "BN + Dropout=0.25": (True, 0.25)
}

task3_results = {}

for name, (bn, dr) in scenarios.items():
    print(f"\n{name}")
    model = MLP([256], use_bn=bn, dropout=dr).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_model(model, optimizer, epochs=10)
    acc = test_model(model)
    task3_results[name] = acc


## Final Comparison Table

In [ ]:
final_table = pd.DataFrame({
    "Experiment": [
        "Sigmoid + SGD",
        "ReLU + SGD",
        "ReLU + Adam"
    ],
    "Final Test Accuracy": [
        task1_results["Sigmoid"]["test_acc"],
        task2_results["SGD"],
        task2_results["Adam"]
    ]
})

final_table
